In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler

class HotelBookingEDA:

    def __init__(self, dataset_path):
        self.df = pd.read_csv(dataset_path)
        
    def header(self, heading):
        print("-"*70)
        print(f"{heading:^70}")
        print("-"*70)

    def load_dataset(self):
        self.header("First 10 Records")
        display(self.df.head(10))
        self.header("Last 10 Records")
        display(self.df.tail(10))
        self.header("Shape of Dataset")
        print(self.df.shape)
        self.header("Features in Dataset")
        print(self.df.columns)
        self.header("Numerical Features")
        print(self.df.select_dtypes(include=["number"]).columns)
        self.header("Categorical Features")
        print(self.df.select_dtypes(include=["object"]).columns)
        self.header("Memory Usage")
        display(self.df.memory_usage(deep=True))

    def overview_dataset(self):
        self.header("Info of Dataset")
        print(self.df.info())
        self.header("Describe Dataset")
        display(self.df.describe())
        self.header("Describe Dataset for Categorical Features")
        display(self.df.describe(include="object"))
        self.header("Data type summary")
        display(self.df.dtypes.reset_index())

    def analyze_missing_values(self):
        self.header("Total Missing Values")
        missing_count = self.df.isna().sum()
        display(missing_count[missing_count > 0])
        self.header("Missing Values Percentage")
        missing_prcnt = (self.df.isna().sum()/self.df.shape[0])*100
        display(missing_prcnt[missing_prcnt > 0])

    def visualize_missing_values(self):
        plt.figure(figsize=(10,5))
        missings = self.df.isna()
        self.header("Missing Values Table")
        display(self.df[missings])
        self.header("Missing Values Heatmap")
        sns.heatmap(missings, cbar=True)
        self.header("Missing Values Bar Chart")
        missings.plot(kind="bar", color="red")
        plt.show()

    def analyze_and_handle_duplicates(self):
        duplicates = self.df[self.df.duplicated()]
        self.header("Duplicates Table")
        display(duplicates)
        self.header("Duplicates Count")
        print(duplicates.shape[0])
        self.header("Removing Duplicates from dataset...")
        self.df.drop_duplicates(inplace=True)
        print("Duplicates Removed :)")

    def analyze_unique_values(self):
        self.header("Unique Values Count")
        display(self.df.nunique().reset_index().rename(columns={0: "Count"}))
        self.header("Most Frequent Value")
        display(self.df.mode())
        self.header("Frequency Percentage")
        freq_prcnt = (self.df.nunique()/self.df.nunique().sum())*100
        display(freq_prcnt.reset_index().rename(columns={0: "Percentage"}))

    def handle_missing_values(self):
        # handle Numerical features
        self.header("Handling Missing Data...")
        numerical_features = self.df.select_dtypes(include=["number"]).columns.to_list()
        skewness = self.df[numerical_features].skew()
        # Nomrmal Distributed Features
        normal_features = skewness[abs(skewness) < 1].index
        self.df[normal_features] = self.df[normal_features].fillna(self.df[normal_features].mean())
        # Skewed Features
        skewed_features = skewness[abs(skewness) > 1].index
        self.df[skewed_features] = self.df[skewed_features].fillna(self.df[skewed_features].median())
    
        # handle Categorical features
        categorical_features = self.df.select_dtypes(include=["object"]).columns.to_list()
        missing_prcnt = (self.df[categorical_features].isna().sum()/self.df[categorical_features].shape[0])*100    
        # High volume missigness
        high_features = missing_prcnt[missing_prcnt > 5].index
        self.df[high_features] = self.df[high_features].fillna("Unknown")
        # Low volume missigness
        low_features = missing_prcnt[(missing_prcnt <= 5) & (missing_prcnt != 0)].index
        self.df[low_features] = self.df[low_features].fillna(self.df[low_features].mode().iloc[0])
        print("Missing Data Handled.")

    def validate_data_types(self):
        self.header("Data Type Validating...")
        # Data type mistakes
        # 1. children & agent & company - float -> int
        self.df[["children", "agent", "company"]] = self.df[["children", "agent", "company"]].astype(int)
        # 2. reservation_status_date - object -> datetime
        self.df["reservation_status_date"] = self.df["reservation_status_date"].astype("datetime64[ns]")
        # 3. is_canceled & is_repeated_guest - int -> category
        self.df[["is_canceled", "is_repeated_guest"]] = self.df[["is_canceled", "is_repeated_guest"]].astype("category")
        # 4. arrival_date_year + arrival_date_mont + arrival_date_day_of_month -> arrival_date (datetime)
        self.df["arrival_date_month"] = self.df["arrival_date_month"].map({"July":7, "August":8, "September":9, "October":10, "November":11, "December":12, "January":1, "February":2, "March":3, "April":4, "May":5, "June":6})
        self.df["arrival_date"] = pd.to_datetime(self.df["arrival_date_year"].astype(str) + "-" + self.df["arrival_date_month"].astype(str) + "-" + self.df["arrival_date_day_of_month"].astype(str))
        # now drop unnecessary features 
        self.df = self.df.drop(columns=["arrival_date_year", "arrival_date_month", "arrival_date_day_of_month"])
        print("Data Validation is complete.")

    def analyze_numerical_features(self):
        self.header("Calculations for Numerical Features")
        numerical_features = self.df.select_dtypes(include=["number"])
        calculations = numerical_features.agg(["mean", "median", "var", "std", "min", "max", "skew"]).T
        calculations["mode"] = numerical_features.mode().iloc[0] 
        calculations["range"] = calculations["max"] - calculations["min"]
        calculations["Q1"] = numerical_features.quantile(0.25)
        calculations["Q3"] = numerical_features.quantile(0.75)
        display(calculations)
        
        self.header("Visualization for Numerical Features")
        numerical_features.hist(figsize=(18, 15))
        plt.suptitle("Histogram", fontsize=24, fontweight="bold")
        plt.show()

        for numerical_feature in numerical_features.columns:
            plt.figure(figsize=(10, 4))
            sns.kdeplot(data=self.df, x=numerical_feature)
            plt.title(f"Density of {numerical_feature}")
            plt.show()

        for numerical_feature in numerical_features.columns:
            plt.figure(figsize=(8, 3))
            sns.boxplot(data=self.df, x=numerical_feature)
            plt.title(f"Boxplot of {numerical_feature}")
            plt.show()

    def analyze_categorical_features(self):
        self.header("Calculations for Categorical Features")
        categorical_cols = self.df.select_dtypes(include=["category"])
        for col in categorical_cols:
            frequency_count = self.df[col].value_counts().reset_index()
            display(frequency_count)
            
        for col in categorical_cols:
            prcnt = round(self.df[col].value_counts(normalize=True) * 100, 2).reset_index()
            prcnt = prcnt.rename(columns={"proportion": "percentage (%)"})
            display(prcnt)

        for col in categorical_cols:
            plt.figure(figsize=(5, 4))
            sns.countplot(data=self.df, x=col)
            plt.show()

        for col in categorical_cols:
            self.df[col].value_counts().plot.pie(autopct="%1.1f%%", startangle=90)
            plt.ylabel("")
            plt.title(f"Distribution of {col}")
            plt.show()

    def analyze_sessional_booking(self):
        self.header("Sessional Booking Analysis")
        months_order = ["January", "February", "March", "April", "May", "June", 
                        "July", "August", "September", "October", "November", "December"]

        self.df["month_name"] = pd.Categorical(self.df["arrival_date"].dt.month_name(), categories=months_order, ordered=True)

        monthly_trend = self.df.groupby("month_name", observed=False).size()
        plt.figure(figsize=(10, 5))
        sns.lineplot(x=monthly_trend.index, y=monthly_trend.values)
        plt.xticks(monthly_trend.index)
        plt.title("Monthly Booking Trend")
        plt.show()
        
        yearly_trend = self.df.groupby(self.df["arrival_date"].dt.year, observed=False).size()
        sns.lineplot(x=yearly_trend.index, y=yearly_trend.values)
        plt.xticks(yearly_trend.index)
        plt.title("Yearly Booking Trend")
        plt.show()

        temp_df = pd.DataFrame({
                "Year": self.df["arrival_date"].dt.year, 
                "Month": pd.Categorical(
                    self.df["arrival_date"].dt.month_name(), 
                    categories=months_order, 
                    ordered=True
                )
        })
        
        heatmap_data = temp_df.groupby(["Year", "Month"], observed=False).size().unstack()
        plt.figure(figsize=(14, 6))
        sns.heatmap(heatmap_data, annot=True, fmt=".0f", linewidths=.5, cbar_kws={"label": "Total Bookings"})
        plt.title("Month-Year Heatmap of Bookings")
        plt.xticks(rotation=45)
        plt.yticks(rotation=0)
        plt.show()

    def detect_outliers(self):
        numerical_features = self.df.select_dtypes(include=["number"])
        self.header("Outliers Detection")
        Q1 = numerical_features.quantile(0.25)
        Q3 = numerical_features.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - IQR * 1.5
        upper_bound = Q3 + IQR * 1.5

        outliers_count = ((numerical_features < lower_bound) | (numerical_features > upper_bound)).sum()
        
        outliers_summary = pd.DataFrame({"Q1": Q1, "Q3": Q3, "IQR": IQR, "Lower Bound": lower_bound, "Upper Bound": upper_bound, "Outliers Count": outliers_count})
        display(outliers_summary)

        for col in numerical_features:
            plt.figure(figsize=(8, 2))
            sns.boxplot(x=self.df[col])
            plt.title(f"Boxplot of {col}")
            plt.xlabel("")
        plt.show()

        # z-score
        z_scores = (numerical_features - numerical_features.mean()) / numerical_features.std()

        outliers_count = (z_scores > 3).sum()[outliers_count > 0].reset_index()
        outliers_count.columns = ["Feature", "Outliers Count"]
        display(outliers_count)

    def handle_outliers(self):
        self.header("Handling Outliers...")
        # Dataset have important and extreme outliers
        # Winsorization
        features = ["lead_time", "stays_in_weekend_nights", "stays_in_week_nights"]
        upper_limit = self.df[features].quantile(0.90)
        self.df[features] = self.df[features].clip(upper=upper_limit, axis=1)
        # Realistic values for family size
        self.df["adults"] = self.df["adults"].clip(upper=4)
        self.df["children"] = self.df["children"].clip(upper=3)
        self.df["babies"] = self.df["babies"].clip(upper=3)
        self.df["adr"] = self.df["adr"].clip(lower=0, upper=1000)
        print("Outliers Handled Successfuly!")

    def feature_engineering(self):
        self.header("Creating New Features...")
        self.df["total_guests"] = self.df["adults"] + self.df["children"] + self.df["babies"]
        self.df["stay_duration"] = self.df["stays_in_weekend_nights"] + self.df["stays_in_week_nights"]
        def assign_season(month):
            if month in [11, 12, 1, 2]:
                return "Winter"
            elif month in [3, 4, 5, 6]:
                return "Summer"
            else:
                return "Monsoon"
                
        self.df["booking_season"] = self.df["arrival_date"].dt.month.map(assign_season)
        def define_guest_type(row):
            if row["children"] > 0 or row["babies"] > 0:
                return "Family"
            elif row["adults"] == 1:
                return "Solo"
            elif row["adults"] == 2:
                return "Couple"
            else:
                return "Family"
        self.df["guest_type"] = self.df.apply(define_guest_type, axis=1)
        self.df["revenue_opportunity"] = self.df["adr"] * self.df["stay_duration"]
        adr_90_prntl = self.df["adr"].quantile(0.90)
        self.df["high_value_customer"] = (self.df["adr"] >= adr_90_prntl).astype(int)
        display(self.df[["total_guests", "stay_duration", "booking_season", "guest_type", "revenue_opportunity", "high_value_customer"]])

    def analyze_bivariate(self):
        self.header("Bivariate Analysis")
        numerical_features = self.df.select_dtypes(include=["number"])
        categorical_features = self.df.select_dtypes(include=["category"])

        self.header("Numerical vs Numerical")
        plt.figure(figsize=(16, 20))
        sns.heatmap(numerical_features.corr(), annot=True, fmt=".2f")
        plt.title("Correlation Heatmap of Numerical Features")
        plt.show()

        for col1 in numerical_features:
            for col2 in numerical_features:
                plt.figure(figsize=(10, 6))
                sns.scatterplot(data=self.df, x=col1, y=col2)
                plt.title(f"{col1} vs {col2}")
                plt.show()
        self.header("Categorical vs Numerical")
        for cat_col in categorical_features:
            for num_col in numerical_features:
                fig, axes = plt.subplots(1, 2, figsize=(18, 6))
                sns.boxplot(data=self.df, x=cat_col, y=num_col, ax=axes[0])
                sns.set_theme(style="whitegrid")
                axes[0].set_title(f"Boxplot: {num_col} by {cat_col}")
                sns.violinplot(data=self.df, x=cat_col, y=num_col, ax=axes[1])
                axes[1].set_title(f"Violinplot: {num_col} by {cat_col}")
                plt.show()
        self.header("Categorical vs Categorical")
        for col1 in categorical_features:
            for col2 in categorical_features:
                crosstab = pd.crosstab(self.df[col1], self.df[col2], margins=True)
                display(crosstab)
                crosstab_plot = pd.crosstab(self.df[col1], self.df[col2])
                plt.figure(figsize=(10, 6))
                sns.heatmap(crosstab_plot, annot=True, fmt="d")
                plt.title(f"Heatmap: {col1} vs {col2}")
                plt.xlabel(col2)
                plt.ylabel(col1)
                plt.xticks(rotation=45)
                plt.yticks(rotation=0)
                plt.show()

    def data_transformation(self):
        self.header("Data Transformation & Encoding")
        # Log transformation 
        skewed_features = ["lead_time", "stays_in_weekend_nights", "stays_in_week_nights"]
        for col in skewed_features:
            if col in self.df.columns:
                self.df[col + "_log"] = np.log1p(self.df[col])
        print(f"Applied Log Transformation to: {skewed_features}")

        # Standardization
        std_scaler = StandardScaler()
        continuous_features = ["adr"]
        if "adr" in self.df.columns:
            self.df[["adr_scaled"]] = std_scaler.fit_transform(self.df[["adr"]])
        print("Applied Standard Scaling to: adr")

        # Min Max Scaling
        minmax_scaler = MinMaxScaler()
        bounded_features = ["adults", "children", "babies", "previous_cancellations", "booking_changes"]
        self.df[bounded_features] = minmax_scaler.fit_transform(self.df[bounded_features])
        print(f"Applied Min-Max Scaling to: {bounded_features}")

        # One Hot Encoding
        low_card_cats = ["hotel", "meal", "market_segment", "distribution_channel", 
                         "deposit_type", "customer_type", "guest_type"]
        self.df = pd.get_dummies(self.df, columns=low_card_cats, drop_first=True)
        print(f"Applied One-Hot Encoding to: {low_card_cats}")
        print("Transformation Complete :)")

    def data_quality_check(self):
        self.header("Data Quality Check Start...")
        def quality_check(col, not_failed):
            if not_failed:
                print(f"{col} QUALITY CHECK PASS :)")
            else:
                print(f"{col} QUALITY CHECK FAILED :(")
        quality_check("adr", self.df["adr"].isnull().sum())
        quality_check("hotel", self.df["hotel"].isnull().sum())
        quality_check("lead_time", (self.df.get("lead_time", 0) < 0).sum())
        quality_check("adr", (self.df.get("adr", 0) <= 0).sum())
        quality_check("adults", (self.df.get("adults", 0) < 0).sum())
        total_guests = self.df.get("adults", 0) + self.df.get("children", 0) + self.df.get("babies", 0)
        quality_check("Total Guests", (total_guests <=0).sum())

    def export_cleaned_dataset(self):
        self.header("Exporting Cleaned Dataset...")
        print("Please Make sure you Executed all the handling and cleaning methods.")
        self.df.to_csv("cleaned_hotel_bookings.csv", index=False)
        print("Cleaned Dataset successfuly exported to cleaned_hotel_bookings.csv")

In [8]:
hotel_data = HotelBookingEDA("/kaggle/input/datasets/jessemostipak/hotel-booking-demand/hotel_bookings.csv")

# Use methods on your preferences